# GitHub Triager RL Training
## Training Llama-3.2-3B to triage GitHub issues using GRPO + TRL + Unsloth

**Environment:** https://huggingface.co/spaces/Kavya011/github-triager-rl  
**GitHub:** https://github.com/KavyaTejani/Github-Triager

In [ ]:
!pip install unsloth trl  transformers datasets peft accelerate matplotlib nest_asyncio
!git clone https://github.com/KavyaTejani/Github-Triager.git
import sys
sys.path.append("/content/Github-Triager")

In [ ]:
ENVIRONMENT_URL = "https://kavya011-github-triager-rl.hf.space"
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct"
MAX_STEPS       = 200      # increase to 500+ for real training
BATCH_SIZE      = 4
NUM_GENERATIONS = 4        # GRPO rollouts per prompt

In [ ]:
from client import GitHubTriagerClient

ENVIRONMENT_URL = "https://kavya011-github-triager-rl.hf.space"

def test_connection():
  with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
    result = env.reset(task_id="label_classification")
    print("✅ Successfully connected to the environment!")
    print("Observation keys:", list(result.keys()))
    return result

obs = test_connection()
print(obs)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded successfully.")

In [ ]:
def rollout_single(task_id: str = "label_classification"):
  with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
    obs = env.reset(task_id=task_id)
    prompt = f"""You are a GitHub issue triager.
Issue Title: {obs['issue']['title']}
Issue Body: {obs['issue']['body']}
Task: Classify this issue. Respond with valid JSON only.
Format: {{"label": "<bug|feature|documentation|question|enhancement>"}}"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):]   # strip prompt from output

    result = env.step({"response": response})
    return prompt, response, float(result.get("reward", 0.0))

# Sanity check
try:
    prompt, response, reward = rollout_single()
    print(f"Reward: {reward:.3f}")
    print(f"Model response: {response}")
except Exception as e:
    print(f"Rollout failed (expected if not on GPU): {e}")

In [ ]:
import json
import re
import requests

def get_env_reward(completion: str) -> float:
  try:
    match = re.search(r'\{.*\}', str(completion), re.DOTALL)
    if match:
      action_data = json.loads(match.group(0))
    else:
      return 0.01 # Model didn't output JSON
  except:
    return 0.01


  try:
    url = f"{ENVIRONMENT_URL}/grade/label_classification"
    response = requests.post(url, json={"action": action_data}, timeout=10)

    if response.status_code == 200:
      return float(response.json().get("score", 0.01))

  except:
    pass
  return 0.01

def compute_reward(prompts, completions, **kwargs):
  return [get_env_reward(c[0]["content"] if isinstance(c, list) else c) for c in completions]

In [ ]:
from datasets import Dataset

def build_dataset(n_samples: int = 100):
  samples = []
  with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
    for i in range(n_samples):
      obs = env.reset(task_id="label_classification")
      prompt = f"""You are a GitHub issue triager.
Issue Title: {obs['issue']['title']}
Issue Body: {obs['issue']['body']}
Classify this issue. Respond with JSON only.
Format: {{"label": "<bug|feature|documentation|question|enhancement>"}}"""
      samples.append({"prompt": prompt})
  return Dataset.from_list(samples)

dataset = build_dataset(100)
print(f"Dataset ready: {len(dataset)} samples")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./outputs/github-triager-grpo",
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=20,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[compute_reward],
    tokenizer=tokenizer,
)

trainer.train()
print("Training complete.")

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("results", exist_ok=True)

log_history = trainer.state.log_history
steps        = [x["step"]   for x in log_history if "loss"   in x]
losses       = [x["loss"]   for x in log_history if "loss"   in x]
r_steps      = [x["step"]   for x in log_history if "reward" in x]
rewards      = [x["reward"] for x in log_history if "reward" in x]

# Loss curve
plt.figure(figsize=(10, 4))
plt.plot(steps, losses, color="royalblue", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("GitHub Triager — Training Loss (GRPO)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/loss_curve.png", dpi=150)
plt.show()
print("Saved: results/loss_curve.png")

# Reward curve
plt.figure(figsize=(10, 4))
plt.plot(r_steps, rewards, color="seagreen", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Average Reward")
plt.title("GitHub Triager — Reward During Training (GRPO)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/reward_curve.png", dpi=150)
plt.show()
print("Saved: results/reward_curve.png")

In [ ]:
import random
import matplotlib.pyplot as plt

def evaluate_model(n_episodes=20):
  total = 0.0
  for _ in range(n_episodes):
    prompt = dataset[random.randint(0, len(dataset)-1)]["prompt"]
    inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):]

    total += get_env_reward(response)

  return total / n_episodes

try:
  trained_avg = evaluate_model(20)
  baseline    = 0.10

  plt.figure(figsize=(6, 5))
  plt.bar(["Baseline", "Trained"], [baseline, trained_avg], color=["#ff6b6b", "#51cf66"], edgecolor="black")
  plt.ylabel("Average Reward")
  plt.title("Before vs After GRPO Training")
  plt.savefig("results/before_after_comparison.png", dpi=150)
  plt.show()
  print("Saved: results/before_after_comparison.png")
except Exception as e:
  print(f"Evaluation failed: {e}")

In [ ]:
model.save_pretrained("outputs/github-triager-adapter")
tokenizer.save_pretrained("outputs/github-triager-adapter")
print("Adapter saved. Do NOT upcast 4-bit model and merge — use adapter directly.")

In [ ]:
def test_reasoning(title,body):
  custom_prompt = f"""Analyze the following Github issue step-by-step.
Identify the likely component and the severity.
Finally, provide the triage JSON.

Title: {title}
Body: {body}

Analysis:"""
  inputs = tokenizer(custom_prompt, return_tensors="pt").to("cuda")
  outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
  print(tokenizer.decode(outputs[0], skip_special_tokens=True))

test_reasoning(
    "AttributeError: 'list' object has no attribute 'keys' when loading gemma"
    "Loading gemma-4b raises a misleading error instead of a version requiremnet."
)
